# 01 · Agent core components

To start, an agent = instructions + model + typed output + tools + tracing.

## Step 1: Set up

The key tells OpenAI who I am, and the imports are the parts I'll assemble.

In [42]:
import os
from typing import List

from agents import Agent, ModelSettings, Runner, function_tool, trace  # the OpenAI Agents SDK pieces I need
from dotenv import find_dotenv, load_dotenv                            # reads my API key from .env
from pydantic import BaseModel, ConfigDict                            
from typing_extensions import TypedDict                                # small fixed-shape dicts

load_dotenv(find_dotenv(usecwd=True))  # look for .env here or in a parent folder
print("API key found:", bool(os.environ.get("OPENAI_API_KEY")))  # just True/False

user_input = "learn about AI agents"  

API key found: True


## Step 2: Build the smallest agent

An agent starts as a job description plus a question. Everything after this is added on top.

In [43]:
# Standing rules for the agent: its role + one hard constraint (5 short tasks)
instructions = """
You are a research planning assistant.

**TASK INSTRUCTIONS**
- You will be given a research topic.
- Your task is to provide a plan on how to research this topic.
- Output 5 concise tasks (5 words or less) to your plan.
"""

# The smallest agent: just a name and instructions (default model, no tools)
agent = Agent(
    name="Research Planner",
    instructions=instructions,
)

result = await Runner.run(agent, input=user_input)  # send my topic, wait for the answer
print(result.final_output)                           # plain text back

1. Define AI agents and scope  
2. Read foundational AI textbooks  
3. Study multi-agent systems research  
4. Explore practical agent frameworks  
5. Implement simple agent-based project


## Step 3: Change the temperature

The model picks each next word from a list of likely options. Temperature controls how far it strays from the most likely one.

The model chooses each next word from options with different probabilities, like “Review” 70%, “Study” 20%, “Explore” 10%. At temperature 0, it nearly always picks the top word ("Review"). At temperature 1, it rolls the dice using those odds, so “Study” comes up about 1 time in 5.

Runs at 0, 0, then 1.

In [44]:
# Helper: same agent, only the temperature changes between runs
def make_planner(temperature):
    return Agent(
        name="Research Planner",
        instructions=instructions,
        model="gpt-4.1", 
        model_settings=ModelSettings(temperature=temperature, max_tokens=150),  # max_tokens caps the length
    )

runs = {}  # label -> output, so I can compare them in the next cell
for label, temp in [("run 1 (temp 0.0)", 0.0), ("run 2 (temp 0.0)", 0.0), ("run 3 (temp 1.0)", 1.0)]:
    result = await Runner.run(make_planner(temp), input=user_input)
    runs[label] = result.final_output
    print(f"--- {label} ---")
    print(result.final_output, "\n")

--- run 1 (temp 0.0) ---
1. Review foundational AI agent literature  
2. Explore recent AI agent advancements  
3. Analyze agent architectures and frameworks  
4. Study real-world agent applications  
5. Compare agent types and capabilities 

--- run 2 (temp 0.0) ---
1. Review foundational AI agent literature  
2. Explore recent AI agent advancements  
3. Analyze agent architectures and frameworks  
4. Study real-world agent applications  
5. Compare leading AI agent platforms 

--- run 3 (temp 1.0) ---
1. Review core AI agent concepts  
2. Compare agent architectures  
3. Analyze real-world agent applications  
4. Study learning algorithms used  
5. Explore current research trends 



Compare the runs:

In [45]:
outputs = list(runs.values())
print("Run 1 == Run 2 (both temp 0):", outputs[0] == outputs[1])  # expect True if temp 0 is repeatable
print("Run 1 == Run 3 (temp 0 vs 1):", outputs[0] == outputs[2])  # expect False

# Print only the lines that differ between the temp-0 and temp-1 runs
lines_1 = outputs[0].splitlines()
lines_3 = outputs[2].splitlines()
print("\nLines that changed between run 1 and run 3:")
for a, b in zip(lines_1, lines_3):
    if a != b:
        print(f"  {a!r}\n  -> {b!r}")

Run 1 == Run 2 (both temp 0): False
Run 1 == Run 3 (temp 0 vs 1): False

Lines that changed between run 1 and run 3:
  '1. Review foundational AI agent literature  '
  -> '1. Review core AI agent concepts  '
  '2. Explore recent AI agent advancements  '
  -> '2. Compare agent architectures  '
  '3. Analyze agent architectures and frameworks  '
  -> '3. Analyze real-world agent applications  '
  '4. Study real-world agent applications  '
  -> '4. Study learning algorithms used  '
  '5. Compare agent types and capabilities'
  -> '5. Explore current research trends'


## Step 4: Ask for typed output (this breaks)

Strict output needs every field named ahead of time. A dict's keys aren't known in advance, so its shape can't be described.

The SDK must know the labels before the model answers. In a dict, the numbers are the labels, so they’re unknown until the end.

In [46]:
# Attempt 1: numbered tasks as a dict {1: "...", 2: "..."}
class ResearchPlanModel(BaseModel):
    tasks: dict[int, str]  # open-ended keys -> strict JSON can't describe this shape
    """A list of tasks to perform for research."""


agent = Agent(
    name="Research Planner",
    instructions=instructions,
    model="gpt-4.1",
    model_settings=ModelSettings(temperature=0.0, max_tokens=150),
    output_type=ResearchPlanModel,  # force the answer into this class
)

try:
    result = await Runner.run(agent, input=user_input)
    print(result.final_output)
except Exception as error:  # expecting a UserError about strict JSON; catch it so Run All keeps going
    print(type(error).__name__, ":", error)

UserError : Strict JSON schema is enabled, but the output type is not valid. Either make the output type strict, or wrap your type with AgentOutputSchema(YourType, strict_json_schema=False)


## Step 5: Fix the shape

Move the number inside each item, and every item has the same known fields. Now the answer is data my code can loop over.

In [47]:
# The fix: each task is a small object with fixed fields
class Task(TypedDict):
    id: int           # the number now lives inside the item
    description: str


class ResearchPlanModel(BaseModel):
    tasks: list[Task]  # a list of fixed-shape items = strict-JSON friendly
    """Numbered tasks for research."""

    model_config = ConfigDict(extra="forbid")  # no extra fields allowed (strict mode needs this)


agent = Agent(
    name="Research Planner",
    instructions=instructions,
    model="gpt-4.1",
    model_settings=ModelSettings(temperature=0.0, max_tokens=150),
    output_type=ResearchPlanModel,
)

result = await Runner.run(agent, input=user_input)
print(result.final_output.tasks)  # a real Python list now, not a string to parse
print("Number of tasks:", len(result.final_output.tasks))

[{'id': 1, 'description': 'Review AI agent definitions'}, {'id': 2, 'description': 'Study agent architectures'}, {'id': 3, 'description': 'Explore real-world applications'}, {'id': 4, 'description': 'Compare agent types'}, {'id': 5, 'description': 'Read recent research papers'}]
Number of tasks: 5


## Step 6: Write a better prompt

Think of the prompt as instructions for a newbie: anything I leave vague, they guess.

- Persona: who they are. "You're a senior librarian."
- Delimiters: where the rules start and stop, so they don’t get mixed up with everything else.
- Exact length: a number, not "keep it short." "5 tasks, 7 words or fewer."
- Positive wording: what to do, not what to avoid. "Start with a verb."

In [48]:
# Rewritten prompt: persona + fenced TASK block (delimiter) + exact length + positive wording
instructions_v2 = """
You are a senior research librarian who helps beginners plan how to learn a new topic.

```TASK
- You will receive one research topic.
- Write a research plan of exactly 5 tasks, in a sensible learning order.
- Keep every task to 7 words or fewer.
- Start each task with an action verb.
```
"""

agent = Agent(
    name="Research Planner",
    instructions=instructions_v2,
    model="gpt-4.1",
    model_settings=ModelSettings(temperature=0.0, max_tokens=150),  # temp stays 0
    output_type=ResearchPlanModel,
)

# Run twice and let Python check the rules instead of counting by hand
for run in (1, 2):
    result = await Runner.run(agent, input=user_input)
    tasks = result.final_output.tasks
    print(f"--- run {run}: {len(tasks)} tasks ---")
    for task in tasks:
        words = len(task["description"].split())
        flag = "ok" if words <= 7 else "TOO LONG"
        print(f"{task['id']}. {task['description']}  ({words} words, {flag})")
    print()

--- run 1: 5 tasks ---
1. Define AI agents and their core concepts  (7 words, ok)
2. Explore types of AI agents  (5 words, ok)
3. Study agent architectures and environments  (5 words, ok)
4. Examine real-world applications of AI agents  (6 words, ok)
5. Review current challenges and future directions  (6 words, ok)

--- run 2: 5 tasks ---
1. Define AI agents and their core concepts  (7 words, ok)
2. Explore types of AI agents  (5 words, ok)
3. Study agent architectures and decision processes  (6 words, ok)
4. Examine real-world applications of AI agents  (6 words, ok)
5. Review challenges and future trends in AI  (7 words, ok)



## Step 7: Give it a tool

The model can't run code. It asks for the tool by name, the SDK runs it, and the result goes back to the model.

Think of the model as a newbie with no computer: it can only talk.

- Model: "Please run get_research_sources."
- SDK: runs my Python function.
- SDK: "Here's the result: Wikipedia, Google, YouTube."
- Model: writes the plan using those sources.

In [49]:
# Same prompt style, but step 1 now tells the agent to call my tool
instructions_v3 = """
You are a senior research librarian who helps beginners plan how to learn a new topic.

```TASK
- First, call get_research_sources() to see which sources are available.
- You will receive one research topic.
- Write a research plan of exactly 5 tasks, using only the available sources.
- Keep every task to 7 words or fewer.
- Name the source each task uses.
```
"""


# Each task now also records which source it uses
class Task(TypedDict):
    step: int
    """Task step."""
    research_source: str
    """The source to search."""
    description: str
    """Task description."""


class ResearchPlanModel(BaseModel):
    tasks: list[Task]
    """Numbered tasks for research."""

    model_config = ConfigDict(extra="forbid")


# @function_tool turns a normal function into a tool; the docstring tells the model what it does
@function_tool
def get_research_sources() -> list[str]:
    """Provides a list of research sources."""
    return ["Wikipedia", "Google", "YouTube"]


agent = Agent(
    name="Research Planner",
    instructions=instructions_v3,
    model="gpt-4.1",
    model_settings=ModelSettings(temperature=0.0, max_tokens=300),  # more room: each task now names a source
    output_type=ResearchPlanModel,
    tools=[get_research_sources],  # registering = listing it here
)

# Group this run under one name so I can find it in the dashboard
with trace("agent_core_components"):
    result = await Runner.run(agent, input=user_input)

for task in result.final_output.tasks:
    print(task["step"], f"[{task['research_source']}]", task["description"])

1 [Wikipedia] Read Wikipedia overview on AI agents.
2 [Wikipedia] Explore types of AI agents on Wikipedia.
3 [Google] Search Google for recent AI agent developments.
4 [YouTube] Watch YouTube introduction to AI agents.
5 [YouTube] Find YouTube examples of AI agents applications.


## Step 8: Confirm the tool ran first

The model needs the sources before it can plan with them. The run's log of events shows the order.

In [50]:
# Everything that happened in the run, in order; I expect a tool call before the final message
for item in result.new_items:
    print(type(item).__name__)

ToolCallItem
ToolCallOutputItem
MessageOutputItem


Trace: [OpenAI dashboard](https://platform.openai.com/traces) → "agent_core_components"